In [12]:
# import importlib
import sqlite3

from unga80.config import DATA_DIR
from unga80 import data_collection as dc
from unga80 import database as dbase

db = DATA_DIR / "countries.db"

In [ ]:
# Create table `country` if not exists
# dbase.drop_country()
dbase.create_db()

In [14]:
# Download spreasheet with the urls to the speeches
df = dc.get_video_url_list()

In [ ]:
# Download the transcripts using the speeches urls
dc.download_all_transcripts(df, overwrite=False)

In [ ]:
# Insert initial data
for i, r in df.iterrows():
    start, end = dc.get_seconds_from_str(r["start"], r["end"])
    corpus = dc.get_corpus_from_file(
        country=r["country"], start=start, end=end, path=DATA_DIR / "2025"
    )
    if corpus:
        dbase.insert(country=r["country"], url=r["url"], speech=corpus)

In [7]:
dbase.select_country()

,country,url,full_speech,summary,countries_mentioned,risks,haiku,word
0,Indonesia,https://www.youtube.com/watch?v=DWEwZY_aZ0s,"[Music] His Excellency Mr. Antonio Gutirez, Se...",The speech is a call to action for world leade...,Here are the two separate lists of countries m...,The provided speech highlights several risks a...,Hope in action\nJustice for all humanity\nPeac...,Hope
1,United States of America,https://www.youtube.com/watch?v=wwy1DYCg-Rs,Thank you very much. Very much appreciated. An...,This is a transcript of a speech delivered by ...,Here are the two separate lists of countries m...,This is a transcript of a speech by the Presid...,This is a transcript of a speech delivered by ...,None
2,Brazil,https://www.youtube.com/watch?v=hG8HZJhB4xo,[Applause] Madame President of the General Ass...,The speech discusses several pressing global i...,Here are the two separate lists of countries m...,The speech discusses several risks and challen...,Peaceful hearts unite\nSovereignty's sacred fl...,Democracy
3,Turkey,https://www.youtube.com/watch?v=Fhxljv9fN_w,"Excellency President of UNGA, excellencies hea...",This is a speech delivered by the President of...,Here are the two separate lists of countries m...,The speech is a lengthy address by the Preside...,This is a transcript of a speech delivered by ...,None
4,Jordan,https://www.youtube.com/watch?v=2jXpVteoZ8w,"Madame President, Mr. Secretary General, your ...",The speech addresses the ongoing conflict in t...,"After analyzing the text, I have identified th...",Here are the main risks and challenges discuss...,"Words fall short still\nJustice for all, not j...",Peace
...,...,...,...,...,...,...,...,...
184,Denmark,https://www.youtube.com/watch?v=3SBmArLIG6Q,"Thank you, Mr. President. Mr. President, disti...",The speech emphasizes the importance of the Un...,Here are the two separate lists of countries m...,Here are the main risks and challenges discuss...,Fragile peace fades fast\r\nBlood-soaked earth...,Progress
185,Vanuatu,https://www.youtube.com/watch?v=3UQOs7KcoZ4,"Madame President, Mr. Secretary General, excel...",The speech focuses on the challenges faced by ...,I have analyzed the text and identified the co...,Here are the main risks and challenges discuss...,Resilience in unity\r\nSolidarity in sovereign...,"""Solidarity"""
186,Benin,https://www.youtube.com/watch?v=gRv0NB-zbIM,"Madame President, >> Madame President, United ...",The speech is a statement by the President of ...,"After analyzing the provided text, I have iden...",Here is a concise list of the main risks/conce...,Global voices matter\r\nSmall states act as co...,Peace
187,Timor-Leste,https://www.youtube.com/watch?v=K7GsvqN2bi8,"Your Excellency Analena Burbach, President of ...",The speech is a lengthy address by the Preside...,**Positive mentions:**\n\n1. **Australia** - ...,Here is a concise list of the main risks/conce...,Respect's call\nUnity's guiding light shines b...,Peace


In [15]:
query = """UPDATE countries
        SET iso_3=?
        WHERE country=?;"""
for i, r in df.iterrows():
    # print(r["country"], r["iso_3"])
    with sqlite3.connect(db) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query,
            (
                r["iso_3"],
                r["country"],
            ),
        )
        conn.commit()
        cursor.close()